In [1]:
import re
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import os
import random
from urllib.parse import urljoin, urlparse
import certifi
import glob
import shutil

In [2]:
LPAs_df_smaller = pd.read_csv("LPAs_smaller_df.txt")
LPAs_df_smaller.head()

,Country,Planning_Authority,Technology_Type,Storage_Type,Planning_Application_Reference,URLS_ADVANCED
0,Scotland,Aberdeen City,Battery,Stand-alone Storage,210665/DPP,https://publicaccess.aberdeencity.gov.uk/onlin...
1,Scotland,Aberdeen City,Battery,Stand-alone Storage,220026/DPP,https://publicaccess.aberdeencity.gov.uk/onlin...
2,Scotland,Aberdeen City,Battery,Stand-alone Storage,231336/DPP,https://publicaccess.aberdeencity.gov.uk/onlin...
3,Scotland,Aberdeen City,Battery,Stand-alone Storage,240614/DPP,https://publicaccess.aberdeencity.gov.uk/onlin...
4,Scotland,Aberdeen City,Battery,Stand-alone Storage,231134/DPP,https://publicaccess.aberdeencity.gov.uk/onlin...


In [3]:
LPAs_df_smaller.shape

(189, 6)

In [4]:
LPAs_df_smaller["Planning_Application_Reference"].is_unique

False

In [5]:
session = requests.Session()

session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-GB,en;q=0.5'
})

In [ ]:
#some of the councils had broken certificates, i had to download the intermediate one and insert it for the scraper to work
CERT_DIR = "LPAs_intermediate_certs"         
CUSTOM_CA_BUNDLE = "custom_ca_bundle.crt"

shutil.copyfile(certifi.where(), CUSTOM_CA_BUNDLE)

for fp in sorted(glob.glob(os.path.join(CERT_DIR, "*"))):
    with open(fp, "rb") as inc, open(CUSTOM_CA_BUNDLE, "ab") as out:
        out.write(b"\n")
        out.write(inc.read())
print("Custom CA bundle created:", CUSTOM_CA_BUNDLE)


SSL_BROKEN_CHAIN_HOSTS = {
    "idoxwam.dundeecity.gov.uk",
    "publicaccess.glasgow.gov.uk",
    "publicaccess.southlanarkshire.gov.uk",
    "ercbuildingstandards.eastrenfrewshire.gov.uk",
    "www.eplanning.north-ayrshire.gov.uk",
    "publicaccess.south-ayrshire.gov.uk",
    "planning.inverclyde.gov.uk",
    "pa.eastlothian.gov.uk",
    "planning.westlothian.gov.uk",
    "planning.cne-siar.gov.uk"
}

Custom CA bundle created: custom_ca_bundle.crt


In [7]:
def verify_for(url: str):
    host = urlparse(url).netloc.lower()
    if host in SSL_BROKEN_CHAIN_HOSTS:
        return CUSTOM_CA_BUNDLE
    return True

#wrappers for session calls
def sget(session, url, **kwargs):
    kwargs.setdefault("verify", verify_for(url))
    return session.get(url, **kwargs)

def spost(session, url, **kwargs):
    kwargs.setdefault("verify", verify_for(url))
    return session.post(url, **kwargs)

In [8]:
#extracting base url, as the first path segment differs between LPAs
def get_base_url(url):
    parsed = urlparse(url)
    origin = f"{parsed.scheme}://{parsed.netloc}"
    first_segment = parsed.path.split("/")[1]
    return f"{origin}/{first_segment}/"

In [9]:
def from_advanced_to_project(session, advanced_url, reference):
#loads advanced search page and returns the value of the hidden _csrf input

#getting to the advanced search page
    r = sget(session, advanced_url, timeout =30)
    
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
#extracting crsf
    csrf = soup.select_one('input[name="_csrf"]')
    if not csrf or not csrf.get("value"):
        raise RuntimeError("No _csrf found on advanced search page")
    csrf_value = csrf["value"]


#post url (simulating clicking the 'search' button)
    base_url = get_base_url(advanced_url) 
    post_url = urljoin(base_url, "advancedSearchResults.do?action=firstPage")

    payload = {
        "_csrf": csrf_value,
        "searchCriteria.reference": reference,
        "caseAddressType": "Application",
        "searchType": "Application",
        }

    r_results = spost(session, post_url, data=payload, timeout=30)
    r_results.raise_for_status()

    return r_results.url, r_results.text

In [10]:
#getting from project's mainpage to Documents subpage
def get_documents_url(project_html, current_url):
    soup = BeautifulSoup(project_html, "html.parser")

    a = soup.select_one("a#tab_documents")
    if not a:
        return None
        #raise RuntimeError("Documents tab link not found")

    return urljoin(current_url, a["href"])

In [11]:
def filter_documents(session, documents_url):
    r = sget(session, documents_url, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    form = soup.select_one("form#caseDetailsForm")
    if not form:
        raise RuntimeError("Could not find documents filter form")  #prob redundant?

    csrf = soup.select_one('input[name="_csrf"]')
    if not csrf or not csrf.get("value"):
        raise RuntimeError("No _csrf found on Documents page")
    csrf_value = csrf["value"]

#for checking whether the filter for public comments exists (if it doesn't it means there are no public comments for the project)
    select = soup.select_one('select[name="documentType"]')
    if not select:
        return None, None
    
    filter_value = None

#making the filter more robust: the filter has different names across sites, but it always contains the word "comment"
    for optn in select.find_all("option"):
        raw_val = (optn.get("value") or "").strip()
        if not raw_val:
            continue
        val = raw_val.lower()

        if "comment" in val and "consultee" not in val:
            filter_value = raw_val
            break
    if not filter_value:
        return None, None


#post_url = urljoin(documents_url, form["action"])  - not needed as the url stays the same, only the payload gets updated

    payload = {
        "_csrf": csrf_value,
        "resetFilter": "false",
        "reloadDocTypes": "false",
        "filterType": "documentType",
        "documentType": filter_value,
        }
    
    #post
    r_filtered = spost(session, documents_url, data=payload, headers={"Referer": documents_url}, timeout=30)
    r_filtered.raise_for_status()
    return r_filtered.url, r_filtered.text

In [12]:
def extract_pdfs(documents_html, base_url):
    soup = BeautifulSoup(documents_html, "html.parser")
    table = soup.select_one("table#Documents") 
    if not table:
        raise RuntimeError("Could not find table#Documents")
    
    return [
        urljoin(base_url, a["href"])
        for a in table.find_all("a")
        if a.get("href", "").endswith(".pdf")
    ]

In [13]:
def download_pdfs(urls, session, folder_path, referer_url): #added referer url
    if_downloaded = False

    for count, url in enumerate(urls, 1):  #for avoiding getting blocked
        try:
            filename = os.path.basename(urlparse(url).path)
            file_path = os.path.join(folder_path, filename)

            #added for speeding things up if I have to rerun the code later on
            if os.path.exists(file_path):
                print(f" File already exists, skipping: {filename}")
                if_downloaded = True
                continue

            print(f"Downloading {filename}...")
            
            #using the same session as before (!!!) but making ajustment to headers
            response = sget(session, url, stream=True, timeout=20, headers={"Referer": referer_url, "Accept":"application/pdf,*/*;q=0.8"})
            response.raise_for_status() 
            
            with open(file_path, "wb") as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            
            print("Downloaded.")
            if_downloaded = True
            time.sleep(random.uniform(2, 5))

#for avoiding getting blocked (as I noticed I get blocked after scraping ten files)
            if count % 10 == 0:
                print(f"Downloaded {count} files. Pausing for 60 seconds to reset the firewall.")
                time.sleep(60)
            

            #this only prints and raises error, doesnt retry to download
        except Exception as e:
            print(f"Failed to download {url}: {e}")
            if "429" in str(e):
                print("Too many requests. Pausing for 60 seconds.")
                time.sleep(60)
            continue

    if not if_downloaded:
        raise RuntimeError(f"Total failure. All {len(urls)} PDFs failed to download.")

In [14]:
#base directory for all LPAs projects
base_save_dir = "LPAs_all_projects"
os.makedirs(base_save_dir, exist_ok=True)

total_projects = len(LPAs_df_smaller)
banned_councils =[]

#for rerunning the code, but skipping the projects that were already succesfully downloaded
completed_rows = set()
if os.path.exists("LPAs_downloads_log.csv"):
    previous_log = pd.read_csv("LPAs_downloads_log.csv")
    #skipping files with these statuses
    success_statuses = ['success', 'no_pdfs', 'no_public_comments_filter']  #removed 'missing_url' 
    completed_log = previous_log[previous_log["Status"].isin(success_statuses)]


    completed_rows = set(zip(
        completed_log["Planning Authority"].astype(str).str.strip(),
        completed_log["Reference"].astype(str).str.strip()))

    print(f"Found {len(completed_rows)} previously completed projects. They will be skipped.")
    
    all_results = previous_log.to_dict('records')
else:
    all_results = []



for idx, row in LPAs_df_smaller.iterrows():

    if idx > 0 and idx % 10 == 0:
        pd.DataFrame(all_results).to_csv("LPAs_downloads_log.csv", index=False)


    advanced_url = str(row["URLS_ADVANCED"])
    project_ref = str(row["Planning_Application_Reference"]).strip()
    # for naming the folders
    authority = str(row['Planning_Authority']).strip()

    #403 bit
    if authority in banned_councils:
        print(f"{authority} is banned (Cloudflare detected earlier). Skipping.")
        continue


    print(f"[{idx+1}/{len(LPAs_df_smaller)}] {project_ref}")

    if (authority, project_ref) in completed_rows:
        print("Processed in a previous run. Skipping.")
        continue


    safe_reference = str(project_ref).replace("/", "_").replace("\\", "_")
    safe_authority = authority.replace(" ", "_")
    folder_name = f"{safe_authority}_{safe_reference}"
  
    try:
        #from advanced search page to projects page
        url, html = from_advanced_to_project(session, advanced_url, project_ref)
        time.sleep(random.uniform(1, 4))  #add more
        #getting to the documents tab
        documents_url = get_documents_url(html, url)
        if documents_url is None:
            print("No Documents tab available for the project.")
            all_results.append({
                "Planning Authority": authority,
                "Reference": project_ref,
                "Status": "documents_tab_missing",
                "Error": ""
            })
            continue
        time.sleep(random.uniform(4, 12))
        #applying Public Comments filter
        url_comments, html_comments = filter_documents(session, documents_url)
        time.sleep(random.uniform(6, 13))


        if url_comments is None:
            print("No public comments filter within the documents tab available for the project.")
            all_results.append({
                "Planning Authority": authority,
                "Reference": project_ref,
                "Status": "no_public_comments_filter",
                "Error": ""
            })
            time.sleep(random.uniform(2, 5))
            continue
    
        pdf_urls = extract_pdfs(html_comments, url_comments)
        if not pdf_urls:
            print("Filter applied, but no PDFs were found.")
            all_results.append({
            "Planning Authority": authority,
            "Reference": project_ref,
            "Status": "no_pdfs",
            "Error": ""
            })
            time.sleep(random.uniform(2, 4))
            continue

        print(f"Found {len(pdf_urls)} PDFs.")

        project_folder = os.path.join(base_save_dir, folder_name)
        os.makedirs(project_folder, exist_ok=True)

        #dowloading the files
        time.sleep(random.uniform(2,7))
        download_pdfs(pdf_urls, session, project_folder, url_comments)
        print(" Project completed.")

        time.sleep(random.uniform(2,5)) #add more 
        
        all_results.append({
            "Planning Authority": authority,
            "Reference": project_ref,
            "Status": "success",
            "Error": ""
        })

    except requests.exceptions.SSLError as e:
        all_results.append({
            "Planning Authority": authority,
            "Reference": project_ref,
            "Status": "SSL_error",
            "Error": repr(e)
        })
        print(f"Error processing reference {project_ref}: {e}")
        time.sleep(random.uniform(2, 4))
        continue


#for the 403 error at the advanced_page
    except requests.exceptions.HTTPError as e:
        if e.response is not None:
            text = (e.response.text or "").lower()
            if e.response.status_code in (403, 429) and ("cloudflare" in text or "checking your browser" in text or "security verification" in text or "cf-ray" in e.response.headers):
                print(f"403 Error! {authority} blocked by Cloudflare.")
                all_results.append({
                 "Planning Authority": authority,
                "Reference": project_ref,
                "Status": "cloudflare_block",
                "Error": repr(e)
             })
                banned_councils.append(authority)
                continue
        #if it is not cloudflare but still http error:
        all_results.append({
        "Planning Authority": authority,
        "Reference": project_ref,
        "Status": "http_error",
        "Error": repr(e)
    })
        print(f"HTTP error processing reference {project_ref}: {e}")
        time.sleep(random.uniform(2, 10))
        continue
        

    #this has to be the last exception as it it the most generic
    except Exception as e:
        all_results.append({
            "Planning Authority": authority,
            "Reference": project_ref,
            "Status": "exception",
            "Error": repr(e)
        })
        print(f"Error processing reference {project_ref}: {e}")
        time.sleep(random.uniform(2, 10))
        continue



log_df = pd.DataFrame(all_results).drop_duplicates(subset=["Planning Authority","Reference"], keep="last")
log_df.to_csv("LPAs_downloads_log.csv", index=False)

Found 139 previously completed projects. They will be skipped.
[1/189] 210665/DPP
Processed in a previous run. Skipping.
[2/189] 220026/DPP
Processed in a previous run. Skipping.
[3/189] 231336/DPP
Processed in a previous run. Skipping.
[4/189] 240614/DPP
Processed in a previous run. Skipping.
[5/189] 231134/DPP
Processed in a previous run. Skipping.
[6/189] 241197/DPP
Processed in a previous run. Skipping.
[7/189] 240313/DPP
Processed in a previous run. Skipping.
[8/189] APP/2018/0488
Processed in a previous run. Skipping.
[9/189] APP/2018/0525
Processed in a previous run. Skipping.
[10/189] APP/2018/0526
Processed in a previous run. Skipping.
[11/189] APP/2018/2702
Processed in a previous run. Skipping.
[12/189] APP/2019/0373
Processed in a previous run. Skipping.
[13/189] ENQ/2020/0261
Processed in a previous run. Skipping.
[14/189] APP/2021/0936
Processed in a previous run. Skipping.
[15/189] APP/2021/2556
Processed in a previous run. Skipping.
[16/189] APP/2021/2855
Processed in a